In [1]:
# Use only 1 GPU if available
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # Disable progress bars

from chronos import BaseChronosPipeline, Chronos2Pipeline
from datetime import datetime
import torch
import os
from datetime import datetime
from ts_toolkit.utils.chronos_tune_input_generator import chronos_input_generator


torch.cuda.empty_cache()


In [2]:
%load_ext autoreload
%autoreload 2

## Can the model learn all at once - copy paste while still maintaining performance on other datasets? Is there an inherent tradeoff? 
Experiment: finetune on Deterministic Trends + I.I.D. White Noise & Stochastic Drift/Shocks


### Data Generation ###

In [3]:
DISTRIBUTIONS = {
    "Deterministic Trends": ['constant', 'linear', 'exponential', 'staircase'],
    "Harmonic & Periodic": ['sine', 'sawtooth', 'square', 'triangle', 'noise_patch', 'fourier'],
    "Transient Dynamics": ['damped_sine', 'growing_sine', 'chirp'],
    "Autocorrelated Processes": ['ar', 'ma', 'arma', 'sarima', 'garch', 'pink_noise', 'fractional_brownian', 'markov_chain'], 
    "I.I.D. White Noise": ['normal', 'uniform', 'poisson', 'binary', 'student_t', 'lognormal', 'laplace', 'cauchy', 'skew_normal'],
    "Stochastic Drift/Shocks": ['random_walk', 'piecewise_constant', 'gbm', 'intermittent', 'impulse', 'logistic'], 
}

In [4]:
# List of level 1 dataset types, e.g., ["univariate_lagged", "multivariate_lagged"]. If None, all types are used.
dataset_level1 = None
# List of level 2 dataset types, e.g., ["exponential", "noise_patch", "normal", "poisson", "random_walk"]. If None, all types are used.
# all but Autocorrelated Processes
dataset_level2 = DISTRIBUTIONS["Deterministic Trends"] + DISTRIBUTIONS["Harmonic & Periodic"] + DISTRIBUTIONS["Transient Dynamics"] + DISTRIBUTIONS["I.I.D. White Noise"] + DISTRIBUTIONS["Stochastic Drift/Shocks"]

data_path = "/home/nafiseh/GitHub/timeseries-research/data/simpletime/datasets_100k"
train_inputs, validation_inputs = chronos_input_generator(data_path, dataset_level1, dataset_level2, val_size=0.1)


multivariate_lagged/constant: 100000it [00:19, 5145.41it/s]
multivariate_lagged/linear: 100000it [00:19, 5211.04it/s]
multivariate_lagged/exponential: 100000it [00:19, 5243.55it/s]
multivariate_lagged/staircase: 100000it [00:18, 5316.63it/s]
multivariate_lagged/sine: 100000it [00:19, 5063.07it/s]
multivariate_lagged/sawtooth: 100000it [00:19, 5072.50it/s]
multivariate_lagged/square: 100000it [00:19, 5175.16it/s]
multivariate_lagged/triangle: 100000it [00:19, 5089.98it/s]
multivariate_lagged/noise_patch: 100000it [00:19, 5085.45it/s]
multivariate_lagged/fourier: 100000it [00:20, 4995.89it/s]
multivariate_lagged/damped_sine: 100000it [00:19, 5117.89it/s]
multivariate_lagged/growing_sine: 100000it [00:19, 5262.56it/s]
multivariate_lagged/chirp: 100000it [00:19, 5137.73it/s]
multivariate_lagged/normal: 100000it [00:19, 5071.52it/s]
multivariate_lagged/uniform: 100000it [00:20, 4810.52it/s]
multivariate_lagged/poisson: 100000it [00:21, 4714.54it/s]
multivariate_lagged/binary: 100000it [00:1


Prepared training inputs for 7560000 training time series and 840000 validation time series.

Sample time series:
  Target shape: (256,)


### Fine-Tuning API

The `fit` method accepts:
- `inputs`: Time series for fine-tuning (same format as predict_quantiles)
- `prediction_length`: Forecast horizon for fine-tuning
- `validation_inputs`: Optional validation data (same format as inputs)
- `learning_rate`: Optimizer learning rate (default: 1e-5)
- `num_steps`: Number of training steps (default: 1000)
- `batch_size`: Batch size for training (default: 256)

Returns a new pipeline with the fine-tuned model.

In [ ]:
output_dir = f"results/finetuned_models/chronos2/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

print(f"Fine-tuning Chronos-2 model. Output directory: {output_dir}")

# Fine-tune the model
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=48,
    num_steps=100_000,
    learning_rate=1e-4,
    batch_size=256,
    logging_steps=100,
    # finetune_mode="lora",
    output_dir=output_dir,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    validation_inputs=validation_inputs,
    eval_steps=100
) 

/tmp/ipykernel_1023940/786704418.py:7: FutureWarning: Fine-tuning support is experimental and may be changed in future versions.
  finetuned_pipeline = pipeline.fit(
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return torch._C._get_cublas_allow_tf32()
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,Validation Loss
100,3.487000,2.899347
200,3.097800,2.288848
300,2.860700,2.290271
400,2.729800,2.203520
500,2.671600,2.201877
600,2.719500,2.215833
700,2.685100,2.165779
800,2.574400,2.140641
900,2.601600,2.135704
1000,2.649200,2.129123


In [ ]:
output_dir

'results/finetuned_models/chronos2/20260211_192422'

In [ ]:
break

Results written in 'results/finetuned_models/chronos2/20260212_115514'

In [ ]:
from ts_toolkit.pipeline.visualization import finetuning_curves

finetuning_curves(output_dir)

In [ ]:
from ts_toolkit.pipeline.runner import TsPipeline
from ts_toolkit.pipeline.utils import load_config
from glob import glob
import os

ft_experiment = "All"
output_paths = {}

for multivariate in [True, False]:
    multivariate_str = "Multivariate" if multivariate else "Univariate"
    model_repo = f"results/finetuned_models/chronos2/20260211_155433"
    config_dict = {
        "experiment_name": f"Chronos2-FineTuned-{ft_experiment}-{multivariate_str}-SimpleTime",

        "paths": {
            "storage_path": "data/simpletime/datasets_100",
            "output_path": "results"
        },
        "forecaster": {
            "name": "MedianEnsemble",
            "max_length": 512,
            "models": [
                {
                    "name": "Chronos2",
                    "params": {
                        "repo_id": f"{model_repo}/finetuned-ckpt",
                        "batch_size": 64
                    },
                }
            ],
            "alias": f"Chronos2-FineTuned-{ft_experiment}-{multivariate_str}-SimpleTime"
        },
        "evaluation": {
            "to_univariate": not multivariate,
            "show_plot": False,
            "eval_substr": "dim0"
        }
    }
    
    config = load_config(config_dict=config_dict)
    # --- Set up the evaluation pipeline ---
    pipeline = TsPipeline(config)
    pipeline.run(skip_existing=True, num_plots=1)

    pipeline.get_results()
    output_paths[f"Chronos2 (Finetuned - {ft_experiment}) - {multivariate_str}"] = pipeline.output_path


/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


Evaluating Datasets:   0%|          | 0/108 [00:00<?, ?it/s]

100it [00:00, 946.94it/s]
100it [00:00, 1002.57it/s]
100it [00:00, 929.82it/s]
100it [00:00, 1045.74it/s]
100it [00:00, 963.03it/s]
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/gluonts/ev/aggregations.py:123: RuntimeWarning: invalid value encountered in divide
  return self.partial_result / self.n
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/numpy/core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/home/nafiseh/GitHub/timeser

Evaluating Datasets:   0%|          | 0/108 [00:00<?, ?it/s]

100it [00:00, 1023.92it/s]
100it [00:00, 919.56it/s]
100it [00:00, 895.64it/s]
100it [00:00, 960.66it/s]
100it [00:00, 1039.64it/s]
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/gluonts/ev/aggregations.py:123: RuntimeWarning: invalid value encountered in divide
  return self.partial_result / self.n
100it [00:00, 957.66it/s]
100it [00:00, 1044.73it/s]
100it [00:00, 1037.35it/s]
100it [00:00, 1053.14it/s]
100it [00:00, 1039.18it/s]
100it [00:00, 1053.41it/s]
100it [00:00, 967.97it/s]
100it [00:00, 821.06it/s]
100it [00:00, 1021.33it/s]
100it [00:00, 1033.35it/s]
100it [00:00, 1042.99it/s]
100it [00:00, 954.33it/s]
100it [00:00, 1040.03it/s]
100it [00:00, 966.05it/s]
100it [00:00, 932.27it/s]
100it [00:00, 920.53it/s]
100it [00:00, 963.64it/s]
100it [00:00, 948.98it/s]
100it [00:00, 955.81it/s]
100it [00:00, 1024.38it/s]
100it [00:00, 966.02it/s]
100it [00:00, 923.42it/s]
100it [00:00, 826.08it/s]
100it [00:00, 894.64it/s]
100it [00:00, 904.90it/s]
100it [00:

In [ ]:
output_paths

{'Chronos2 (Finetuned - All) - Multivariate': '/home/nafiseh/GitHub/timeseries-research/results/Chronos2-FineTuned-All-Multivariate-SimpleTime_20260211-161928',
 'Chronos2 (Finetuned - All) - Univariate': '/home/nafiseh/GitHub/timeseries-research/results/Chronos2-FineTuned-All-Univariate-SimpleTime_20260211-162356'}